# Глава 32. Кластеризация на практике

_Ноутбук сгенерирован из учебника «Машинное обучение для начинающих». Запускайте ячейки по порядку сверху вниз в Google Colab._

**Цель главы  🎯**

После этой главы ты сможешь применить K-Means для разбиения данных на группы, выбрать число кластеров методом локтя и понимать, когда нужен DBSCAN вместо K-Means.

В самой первой главе мы упомянули кластеризацию — обучение без учителя, где модель сама находит группы в данных без готовых ответов. Пришло время реализовать её. Это мощный инструмент, когда ты не знаешь заранее, какие группы есть в данных, но подозреваешь, что они существуют.

**Аналогия: рассадить гостей по интересам  💡**

Представь вечеринку, где гости незнакомы. Ты хочешь рассадить их по столам так, чтобы за каждым оказались похожие по интересам люди, — но списка групп у тебя нет. Ты наблюдаешь, кто о чём говорит, и группируешь похожих. Кластеризация делает то же с данными: находит естественные группы похожих объектов, хотя никто не сказал заранее, сколько групп и какие они.

## K-Means: разбить на K групп

Самый популярный метод. Ты говоришь, на сколько групп (K) разбить, а алгоритм сам находит центры групп и распределяет объекты к ближайшему центру. Работает быстро и понятно.

In [ ]:
# === Подготовка: данные пингвинов ===
import pandas as pd
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
df = pd.read_csv(url).dropna(subset=["bill_length_mm", "body_mass_g"]).reset_index(drop=True)

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Данные о пингвинах: длина клюва и масса тела
X = df[["bill_length_mm", "body_mass_g"]]
X_scaled = StandardScaler().fit_transform(X)   # масштаб важен!

# Разбиваем на 3 группы
km = KMeans(n_clusters=3, random_state=42, n_init=10)
df["группа"] = km.fit_predict(X_scaled)

print(df.groupby("группа")[["bill_length_mm", "body_mass_g"]].mean())

_Каждому клиенту присвоена группа; смотрим средний портрет каждой группы._

**Внимание  ⚠️**

K-Means измеряет расстояния — как KNN, он чувствителен к масштабу признаков. Всегда масштабируй данные (StandardScaler) перед кластеризацией, иначе признак с большими числами перекроет остальные.

## Сколько групп? Метод локтя

**Аналогия: где график «ломается»  💡**

Как узнать, на сколько групп делить, если K задаём мы сами? Метод локтя: пробуем разные K и смотрим, как падает «разброс внутри групп». Сначала он падает быстро (группы реально разделяются), потом почти перестаёт — график изгибается, как локоть руки. Точка изгиба («локоть») и есть хорошее число групп: дальше дробить смысла мало.

In [ ]:
import matplotlib.pyplot as plt

# Пробуем K от 1 до 8, смотрим разброс внутри групп (inertia)
разброс = []
for k in range(1, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
    разброс.append(km.inertia_)

plt.plot(range(1, 9), разброс, marker="o")
plt.xlabel("Число групп K"); plt.ylabel("Разброс внутри групп")
plt.title("Метод локтя: ищем изгиб")
plt.show()

На графике ищи точку, где кривая резко перестаёт падать — это и есть «локоть». Есть и второй способ — коэффициент силуэта (silhouette), который численно оценивает, насколько чётко разделены группы; чем ближе к 1, тем лучше.

## DBSCAN: для групп странной формы

K-Means любит круглые группы примерно одного размера и требует заранее знать K. Но что, если группы вытянутые, изогнутые или их число неизвестно? Тогда помогает DBSCAN: он находит группы как «плотные скопления точек» любой формы и сам определяет их число, а редкие одиночные точки помечает как шум (выбросы). Отлично подходит для данных сложной формы.

In [ ]:
from sklearn.cluster import DBSCAN

# eps — радиус соседства, min_samples — минимум точек для группы
db = DBSCAN(eps=0.5, min_samples=5)
df["группа"] = db.fit_predict(X_scaled)
# Метка -1 означает "шум" (точка не вошла ни в одну группу)

## Локальный пример: сегментация клиентов банка

Банк в Душанбе хочет понять своих клиентов, не имея готовых категорий. K-Means по признакам (возраст, средний остаток, число операций) выделит естественные группы: например, «молодые активные», «пожилые сберегатели», «премиум-клиенты». Это классическая задача сегментации: группы не заданы заранее, модель находит их сама, а банк потом предлагает каждой группе своё. Именно так работает персонализация в реальном бизнесе.

## Задания

Базовый уровень

Возьми датасет с числовыми признаками (клиенты, страны, товары). Отмасштабируй, примени K-Means с K=3 и опиши словами портрет каждой группы по средним значениям признаков.

Со звёздочкой  ⭐

Для тех же данных построй метод локтя и определи оптимальное K. Затем сравни результат K-Means с DBSCAN. Нашли ли они похожие группы? Пометил ли DBSCAN какие-то точки как шум? Что эти «шумные» точки означают в твоих данных?

Вопросы для обсуждения  💬

Чем кластеризация отличается от классификации (вспомни главу 1)?

Почему перед K-Means обязательно масштабировать признаки?

В каком случае DBSCAN справится там, где K-Means провалится?

**Проверь себя  ℹ️**

1. Что делает алгоритм K-Means?

2. Как метод локтя помогает выбрать число групп?

3. Когда DBSCAN лучше K-Means?